# Batch Gradient Descent (BGD)

In simple linear regression, optimization only involves finding two parameters: a single slope and an intercept. However, real-world datasets have multiple features ($n$-dimensional data), which means we must optimize multiple coefficients simultaneously. **Batch Gradient Descent (BGD)** is the standard gradient descent variant that calculates the loss gradients over the **entire training dataset** before updating the parameters in a single step.

---

## 1. The Three Variants of Gradient Descent

Depending on how much data is processed before updating the model parameters, gradient descent is categorized into three types:

| Variant | Data per Update | Pros | Cons |
| :--- | :--- | :--- | :--- |
| **Batch Gradient Descent (BGD)** | **Entire Dataset** | • Smooth, stable convergence path.<br>• Mathematically guaranteed to converge to the minimum of a convex function. | • Highly computationally expensive for massive datasets.<br>• Requires fitting the entire dataset in memory. |
| **Stochastic Gradient Descent (SGD)** | **Single Row / Sample** | • Extremely fast computations per step.<br>• Fits easily in memory.<br>• Can escape local minima. | • Noisy, zigzagging convergence path.<br>• Never fully settles at the absolute minimum (oscillates around it) without learning rate decay. |
| **Mini-Batch Gradient Descent** | **Predefined Batch Size** (e.g., 32, 64) | • Balances the stability of BGD and speed of SGD.<br>• Leverages vectorized operations in modern hardware. | • Introduces batch size as an additional hyperparameter to tune. |

---

## 2. Mathematical Formulation for N-Dimensional Data

When predicting a target using $m$ features, the multiple linear regression equation is:

$$\hat{y}_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \dots + \beta_m x_{im}$$

Using the Mean Squared Error (MSE) loss function:

$$L(\beta) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

We must calculate the partial derivative of the loss function with respect to each parameter $\beta_j$.

### 2.1 Individual Parameter Gradients
- **For Intercept ($\beta_0$):**
  $$\frac{\partial L}{\partial \beta_0} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)$$

- **For Feature Coefficients ($\beta_j$ where $j \ge 1$):**
  $$\frac{\partial L}{\partial \beta_j} = -\frac{2}{n} \sum_{i=1}^{n} x_{ij} (y_i - \hat{y}_i)$$

---

## 2.2 Vectorized Formulation (Matrix Math)

Calculating the gradients for each coefficient individually using nested loops in Python is extremely slow. Instead, we use **vectorization** with matrix operations to calculate all gradients simultaneously.

Let $X$ be the input matrix of size $n \times (m+1)$ (with a column of $1$s added at index 0), $Y$ be the actual values vector ($n \times 1$), and $\beta$ be the parameter vector of size $(m+1) \times 1$.

The predicted target vector $\hat{Y}$ is:

$$\hat{Y} = X \beta$$

The difference between actual and predicted target values represents the error vector:

$$\text{Error} = Y - X\beta$$

The gradient vector $\nabla L(\beta)$ containing the gradients for all parameters is:

$$\nabla L(\beta) = -\frac{2}{n} X^T (Y - X\beta)$$

### Vectorized Update Rule
We update the entire parameter vector $\beta$ simultaneously:

$$\beta_{new} = \beta_{old} - \eta \cdot \nabla L(\beta)$$

$$\beta_{new} = \beta_{old} + \frac{2\eta}{n} X^T (Y - X\beta)$$

Where $\eta$ is the learning rate and $X^T$ is the transpose of the input feature matrix.

---

## 3. Code Implementation: Vectorized Batch Gradient Descent from Scratch

We will now build a custom `GDRegressor` class from scratch using NumPy to implement this vectorized matrix formula, then verify it against scikit-learn's `LinearRegression` model.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Generate synthetic regression dataset (100 samples, 3 features)
X, y = make_regression(n_samples=150, n_features=3, noise=10, random_state=42)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")


In [ ]:
# Train scikit-learn LinearRegression model
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
r2_lr = r2_score(y_test, y_pred_lr)

print("="*50)
print("SCIKIT-LEARN MODEL RESULTS")
print("="*50)
print(f"R2 Score:    {r2_lr:.6f}")
print(f"Intercept:   {lr.intercept_:.6f}")
print(f"Coefficients: {lr.coef_}")
print("="*50)


In [ ]:
class GDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.lr = learning_rate
        self.epochs = epochs
        self.coef_ = None
        self.intercept_ = None
        
    def fit(self, X_train, y_train):
        # 1. Insert a column of 1s at index 0 to account for intercept
        X_new = np.insert(X_train, 0, 1, axis=1)
        n_samples, n_features = X_new.shape
        
        # 2. Initialize parameter vector (beta) to zeros
        # Size: (n_features, 1) -> (m + 1) rows, 1 column
        beta = np.zeros(n_features)
        
        # 3. Fit loop
        for epoch in range(self.epochs):
            # Prediction: y_pred = X * beta
            y_pred = np.dot(X_new, beta)
            
            # Gradient calculation: d_beta = -2/n * X^T * (y_actual - y_pred)
            # np.dot(X_new.T, y_train - y_pred) does the matrix transpose multiplication
            d_beta = - (2 / n_samples) * np.dot(X_new.T, y_train - y_pred)
            
            # Parameter update: beta = beta - lr * d_beta
            beta = beta - self.lr * d_beta
            
        # 4. Extract parameters
        self.intercept_ = beta[0]  # First element is intercept
        self.coef_ = beta[1:]      # Rest are feature coefficients
        
    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_


In [ ]:
# Train custom GDRegressor model
# Note: Learning rate and epochs must be tuned for optimal convergence
gd = GDRegressor(learning_rate=0.1, epochs=1000)
gd.fit(X_train, y_train)

y_pred_gd = gd.predict(X_test)
r2_gd = r2_score(y_test, y_pred_gd)

print("="*50)
print("CUSTOM BGD MODEL RESULTS")
print("="*50)
print(f"R2 Score:    {r2_gd:.6f}")
print(f"Intercept:   {gd.intercept_:.6f}")
print(f"Coefficients: {gd.coef_}")
print("="*50)

# Verify parameters are extremely close to scikit-learn's baseline
print("DIFFERENCE ANALYSIS:")
print(f"Intercept Diff:  {abs(lr.intercept_ - gd.intercept_):.6f}")
print(f"Coefficients Diff: {np.abs(lr.coef_ - gd.coef_)}")


## 4. Hyperparameter Tuning

Selecting appropriate values for the **Learning Rate** and the number of **Epochs** is critical for gradient descent:
- If the **learning rate is too low**, the model will need a huge number of epochs to converge to scikit-learn's optimal coefficients.
- If the **epochs are too low**, the model will stop updating before reaching the bottom of the loss surface.
- Adjusting these parameters to match standard library outcomes builds a deep practical understanding of optimization dynamics.